<a href="https://colab.research.google.com/github/rous0307/AnaliticaIncidentesVialesCDMX_Galacticas/blob/main/avance02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avance 2: lectura, perfilamiento y preparación de datos
**Proyecto:** Analítica de Incidentes Viales CDMX (C4/C5)   
**Equipo** Las Galácticas  
**Propósito:** leer el dataset crudo, perfilarlo, limpiarlo y validarlo.  
**Fuente:** Kaggle, "Mexico Road Accidents during 2017-2019" (CSV).  
**Archivos crudos(no se modifican):**    
- data/raw/incidentes-viales-c5-2017.csv  
- data/raw/incidentes-viales-c5-2018.csv  
- data/raw/incidentes-viales-c5-2019.csv  

**Fecha de obtención:**  descargados de Kaggle el 04/10/2026.  
**Orden de ejecución:** de arriba hacia abajo.  
**Secciones:** 1. Lectura · 2. Perfilamiento · 3. Limpieza · 4. Validación

## 0. Configuración
Objetivo: conectar Drive y definir las rutas de los archivos

In [81]:
# Montaje de Drive y definición de rutas de los 3 CSV crudos (2017-2019); solo se declaran, aún no se leen.

from google.colab import drive
from pathlib import Path
import pandas as pd


drive.mount('/content/drive')

base = Path("/content/drive/MyDrive/BI S&R/Datos/Kaggle")

rutas = {
    2017: base / "incidentes-viales-c5-2017.csv",
    2018: base / "incidentes-viales-c5-2018.csv",
    2019: base / "incidentes-viales-c5-2019.csv",
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [82]:
# Montaje de Drive y definición de rutas de los 3 CSV (2017-2019)

from google.colab import drive
from pathlib import Path
import pandas as pd
import os

# Conectar Google Drive
drive.mount('/content/drive')

# Carpeta donde están los archivos CSV
base = Path("/content/drive/MyDrive/BI S&R/Datos/Kaggle")

# Definir las rutas
rutas = {
    2017: base / "incidentes-viales-c5-2017.csv",
    2018: base / "incidentes-viales-c5-2018.csv",
    2019: base / "incidentes-viales-c5-2019.csv",
}

# Comprobar que existen los archivos
for anio, ruta in rutas.items():
    print(f"{anio}: {'Encontrado' if ruta.exists() else 'No encontrado'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2017: Encontrado
2018: Encontrado
2019: Encontrado


#**1. Lectura**
Objetivo: leer los tres archivos CSV sin modificar los originales

In [58]:
# Lee los 3 CSV (sep=",", utf-8) y muestra filas, columnas y nombres por año.

tablas = {}
for anio, ruta in rutas.items():
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontro: {ruta}")
    tablas[anio] = pd.read_csv(ruta, sep=",", encoding="utf-8")
    print(anio, tablas[anio].shape)
    print(tablas[anio].columns.tolist())

2017 (226535, 18)
['folio', 'fecha_creacion', 'hora_creacion', 'dia_semana', 'codigo_cierre', 'fecha_cierre', 'año_cierre', 'mes_cierre', 'hora_cierre', 'delegacion_inicio', 'incidente_c4', 'latitud', 'longitud', 'clas_con_f_alarma', 'tipo_entrada', 'delegacion_cierre', 'geopoint', 'mes']
2018 (246577, 18)
['folio', 'fecha_creacion', 'hora_creacion', 'dia_semana', 'codigo_cierre', 'fecha_cierre', 'año_cierre', 'mes_cierre', 'hora_cierre', 'delegacion_inicio', 'incidente_c4', 'latitud', 'longitud', 'clas_con_f_alarma', 'tipo_entrada', 'delegacion_cierre', 'geopoint', 'mes']
2019 (207634, 18)
['folio', 'fecha_creacion', 'hora_creacion', 'dia_semana', 'codigo_cierre', 'fecha_cierre', 'año_cierre', 'mes_cierre', 'hora_cierre', 'delegacion_inicio', 'incidente_c4', 'latitud', 'longitud', 'clas_con_f_alarma', 'tipo_entrada', 'delegacion_cierre', 'geopoint', 'mes']


In [59]:
# Verificación inicial: confirma el tipo de objeto leído (DataFrame) y muestra
# los primeros 3 registros de cada año para comparar con el inventario del avance 1.
# No modifica los datos.

for anio, tabla in tablas.items():
    print(anio, type(tabla))
    print(tabla.head(3))
    print()

2017 <class 'pandas.core.frame.DataFrame'>
             folio fecha_creacion hora_creacion dia_semana  \
0  C5/170622/06952     22/06/2017      20:34:45     Jueves   
1  C5/170622/05115     22/06/2017      16:41:14     Jueves   
2  C5/170622/03715     22/06/2017      13:44:52     Jueves   

                                       codigo_cierre fecha_cierre  año_cierre  \
0  (D) El incidente reportado se registró en dos ...   22/06/2017        2017   
1  (D) El incidente reportado se registró en dos ...   22/06/2017        2017   
2  (N) La unidad de atención a emergencias fue de...   22/06/2017        2017   

  mes_cierre hora_cierre    delegacion_inicio  \
0      Junio    20:39:42           MILPA ALTA   
1      Junio    16:46:02              TLAHUAC   
2      Junio    16:48:07  VENUSTIANO CARRANZA   

                      incidente_c4   latitud  longitud clas_con_f_alarma  \
0  accidente-choque sin lesionados  19.20137 -99.00732        EMERGENCIA   
1  accidente-choque sin lesionados

#**2. Perfilamiento**

Objetivo: conocer los datos sin modificar ningún valor

In [60]:
# Unión de los 3 años en una tabla; se agrega archivo_origen para conservar la trazabilidad.

incidentes = pd.concat(
    [t.assign(archivo_origen=anio) for anio, t in tablas.items()],
    ignore_index=True
)
print(incidentes.shape)

(680746, 19)


In [61]:
# Estructura: primeras y últimas filas, y resumen de tipos y no nulos.

print(incidentes.head(3))
print(incidentes.tail(3))
incidentes.info()

             folio fecha_creacion hora_creacion dia_semana  \
0  C5/170622/06952     22/06/2017      20:34:45     Jueves   
1  C5/170622/05115     22/06/2017      16:41:14     Jueves   
2  C5/170622/03715     22/06/2017      13:44:52     Jueves   

                                       codigo_cierre fecha_cierre  año_cierre  \
0  (D) El incidente reportado se registró en dos ...   22/06/2017        2017   
1  (D) El incidente reportado se registró en dos ...   22/06/2017        2017   
2  (N) La unidad de atención a emergencias fue de...   22/06/2017        2017   

  mes_cierre hora_cierre    delegacion_inicio  \
0      Junio    20:39:42           MILPA ALTA   
1      Junio    16:46:02              TLAHUAC   
2      Junio    16:48:07  VENUSTIANO CARRANZA   

                      incidente_c4   latitud  longitud clas_con_f_alarma  \
0  accidente-choque sin lesionados  19.20137 -99.00732        EMERGENCIA   
1  accidente-choque sin lesionados  19.29659 -99.03443        EMERGENCIA   
2

In [62]:
# Perfil por columna: tipo, valores presentes, nulos, valores distintos y % de nulos.

perfil = pd.DataFrame({
    "tipo_detectado": incidentes.dtypes.astype(str),
    "valores_presentes": incidentes.notna().sum(),
    "valores_nulos": incidentes.isna().sum(),
    "valores_distintos": incidentes.nunique(dropna=True)
})
perfil["porcentaje_nulos"] = (perfil["valores_nulos"] / len(incidentes) * 100).round(3)
print(perfil)

                  tipo_detectado  valores_presentes  valores_nulos  \
folio                     object             680746              0   
fecha_creacion            object             680746              0   
hora_creacion             object             680746              0   
dia_semana                object             680746              0   
codigo_cierre             object             680746              0   
fecha_cierre              object             680746              0   
año_cierre                 int64             680746              0   
mes_cierre                object             680746              0   
hora_cierre               object             680746              0   
delegacion_inicio         object             680687             59   
incidente_c4              object             680746              0   
latitud                  float64             680303            443   
longitud                 float64             680311            435   
clas_con_f_alarma   

In [63]:
# Duplicados: filas exactas (sin archivo_origen) y repeticiones de folio; solo se muestran.

cols_originales = [c for c in incidentes.columns if c != "archivo_origen"]
exactos = incidentes.duplicated(subset=cols_originales).sum()
folios_rep = incidentes.loc[incidentes.duplicated(subset=["folio"], keep=False)]

print("duplicados exactos:", exactos)
print("filas con folio repetido:", len(folios_rep))
print("folios distintos repetidos:", folios_rep["folio"].nunique())
print(folios_rep.sort_values("folio")[
    ["folio", "fecha_creacion", "hora_creacion", "delegacion_inicio", "archivo_origen"]
].head(6))

duplicados exactos: 0
filas con folio repetido: 0
folios distintos repetidos: 0
Empty DataFrame
Columns: [folio, fecha_creacion, hora_creacion, delegacion_inicio, archivo_origen]
Index: []


In [64]:
# Frecuencias de columnas categóricas (conserva los nulos con dropna=False).

cats = ["dia_semana", "delegacion_inicio", "incidente_c4",
        "codigo_cierre", "clas_con_f_alarma", "tipo_entrada"]
for c in cats:
    print(incidentes[c].value_counts(dropna=False))
    print()

dia_semana
Viernes      113396
Sábado       108599
Jueves        96602
Miércoles     93664
Martes        91323
Lunes         89267
Domingo       87895
Name: count, dtype: int64

delegacion_inicio
IZTAPALAPA             111344
GUSTAVO A. MADERO       69597
CUAUHTEMOC              67755
MIGUEL HIDALGO          56819
COYOACAN                54124
BENITO JUAREZ           52070
ALVARO OBREGON          51760
VENUSTIANO CARRANZA     45682
TLALPAN                 43862
AZCAPOTZALCO            29146
IZTACALCO               27741
XOCHIMILCO              26409
TLAHUAC                 16889
CUAJIMALPA              13774
MAGDALENA CONTRERAS     10143
MILPA ALTA               3572
NaN                        59
Name: count, dtype: int64

incidente_c4
accidente-choque sin lesionados                  358734
accidente-choque con lesionados                  163719
lesionado-atropellado                             90671
accidente-motociclista                            41333
accidente-volcadura           

### Revisión de fechas
Se convierte `fecha_creacion` a fecha en una versión temporal (`fecha_tmp`), sin modificar la tabla, para ver si alguna falla, cuál es el rango real de fechas y de qué año son las fechas de cada archivo.

In [65]:
# Conversión temporal de fecha_creacion (formatos dd/mm/aaaa y dd/mm/aa) sin modificar la tabla.

f4 = pd.to_datetime(incidentes["fecha_creacion"], format="%d/%m/%Y", errors="coerce")
f2 = pd.to_datetime(incidentes["fecha_creacion"], format="%d/%m/%y", errors="coerce")
fecha_tmp = f4.fillna(f2)

fallos = incidentes["fecha_creacion"].notna() & fecha_tmp.isna()
print("fallos de conversión:", fallos.sum())
print("mínima:", fecha_tmp.min(), "| máxima:", fecha_tmp.max())
print(pd.crosstab(incidentes["archivo_origen"], fecha_tmp.dt.year))

fallos de conversión: 0
mínima: 2016-12-07 00:00:00 | máxima: 2019-11-30 00:00:00
fecha_creacion  2016    2017    2018    2019
archivo_origen                              
2017              19  226516       0       0
2018               0      14  246563       0
2019               0       0      16  207618


### Formato de las fechas por archivo
Se cuentan los días distintos que hay de verdad y se revisa cómo viene escrito el año de `fecha_creacion` en cada archivo: con 2 dígitos (8 caracteres) o con 4 (10 caracteres).

In [66]:
# Longitud del texto de fecha_creacion por archivo (8 = dd/mm/aa, 10 = dd/mm/aaaa) y fechas reales distintas.

print("fechas reales distintas:", fecha_tmp.nunique())
print(pd.crosstab(incidentes["archivo_origen"], incidentes["fecha_creacion"].str.len()))

fechas reales distintas: 1066
fecha_creacion      8       10
archivo_origen                
2017                 0  226535
2018                 0  246577
2019            109047   98587


### Variables numéricas
Resumen de `latitud`, `longitud`, `año_cierre` y `mes` (mínimo, máximo y cuartiles) para detectar valores raros.

In [67]:
# Resumen descriptivo de las columnas numéricas (mínimo, máximo, cuartiles).

print(incidentes[["latitud", "longitud", "año_cierre", "mes"]].describe())

             latitud       longitud     año_cierre            mes
count  680303.000000  680311.000000  680746.000000  680746.000000
mean       19.383244     -99.150441    2017.972235       6.565763
std         0.373527       3.420512       0.798132       3.384557
min        19.095427    -991.764000    2017.000000       1.000000
25%        19.336040     -99.180420    2017.000000       4.000000
50%        19.383030     -99.140680    2018.000000       7.000000
75%        19.433760     -99.095230    2019.000000      10.000000
max       195.303000     -98.945370    2019.000000      12.000000


In [68]:
# Coordenadas fuera del recuadro aproximado (cada una por separado) y mes vs mes de fecha_cierre.
fuera_lat = incidentes["latitud"].notna() & ~incidentes["latitud"].between(19.0, 19.6)
fuera_lon = incidentes["longitud"].notna() & ~incidentes["longitud"].between(-99.4, -98.9)
print("latitud fuera:", fuera_lat.sum(), "| longitud fuera:", fuera_lon.sum())
print(incidentes.loc[fuera_lat | fuera_lon,
      ["folio", "latitud", "longitud", "geopoint", "archivo_origen"]].head(10))

c4 = pd.to_datetime(incidentes["fecha_cierre"], format="%d/%m/%Y", errors="coerce")
c2 = pd.to_datetime(incidentes["fecha_cierre"], format="%d/%m/%y", errors="coerce")
cierre_tmp = c4.fillna(c2)
print("mes distinto al mes de fecha_cierre:", (incidentes["mes"] != cierre_tmp.dt.month).sum())

latitud fuera: 3 | longitud fuera: 10
                  folio  latitud  longitud geopoint  archivo_origen
347352  C5/181231/05744      NaN  -990.495      NaN            2018
347369  C5/181231/00956      NaN  -991.764      NaN            2018
347384  C5/181231/04311      NaN  -989.595      NaN            2018
347458  GA/181231/04533  195.303       NaN      NaN            2018
359372  C5/181231/01210      NaN  -990.369      NaN            2018
359387  C5/181231/05350      NaN  -991.692      NaN            2018
359407  C5/181231/06094  193.815       NaN      NaN            2018
452657  C5/181231/06210      NaN  -990.765      NaN            2018
452674  C5/181231/06971  193.446       NaN      NaN            2018
452683  C5/181231/01068      NaN  -991.764      NaN            2018
mes distinto al mes de fecha_cierre: 0


### Comprobación: alcaldías vacías y valores "NA"
Se cuentan por archivo los valores nulos de `delegacion_inicio`, que juntan las celdas vacías y las que dicen "NA" (Pandas las lee igual), y se muestran cinco filas de ejemplo.

In [69]:
# Filas con delegacion_inicio vacía: distribución por archivo y últimas filas.
vacias = incidentes[incidentes["delegacion_inicio"].isna()]
print(vacias["archivo_origen"].value_counts())
print(vacias.tail(5)[["folio", "fecha_creacion", "delegacion_inicio",
                      "delegacion_cierre", "latitud", "longitud", "archivo_origen"]])


archivo_origen
2018    36
2019    19
2017     4
Name: count, dtype: int64
                  folio fecha_creacion delegacion_inicio delegacion_cierre  \
621892  GA/190414/02002       14/04/19               NaN               NaN   
644678  C5/190514/00119       14/05/19               NaN               NaN   
653644  C5/190706/01549       06/07/19               NaN               NaN   
659625  C5/191017/07733     17/10/2019               NaN               NaN   
670217  C5/190728/08293       28/07/19               NaN               NaN   

          latitud   longitud  archivo_origen  
621892  19.442927 -99.077489            2019  
644678  19.418199 -99.079211            2019  
653644  19.427160 -99.073190            2019  
659625  19.435360 -99.086930            2019  
670217  19.270770 -98.960160            2019  


Se vuelven a leer los CSV sin convertir "NA" en nulo, para separar las celdas vacías de las que dicen "NA" en `delegacion_inicio` y `delegacion_cierre`. No se modifica ningún dato.

In [70]:
# Comprobación: leer sin convertir "NA" en nulo para distinguir celdas vacías de "NA" literal.
for anio, ruta in rutas.items():
    crudo = pd.read_csv(ruta, sep=",", encoding="utf-8", keep_default_na=False, dtype=str)
    for col in ["delegacion_inicio", "delegacion_cierre"]:
        print(anio, col, "| vacías:", (crudo[col] == "").sum(), "| NA:", (crudo[col] == "NA").sum())

2017 delegacion_inicio | vacías: 4 | NA: 0
2017 delegacion_cierre | vacías: 4 | NA: 0
2018 delegacion_inicio | vacías: 36 | NA: 0
2018 delegacion_cierre | vacías: 18 | NA: 0
2019 delegacion_inicio | vacías: 16 | NA: 3
2019 delegacion_cierre | vacías: 16 | NA: 3


## 3. Limpieza y normalización
Objetivo: aplicar reglas justificadas sobre una copia (`limpio`) de la tabla `incidentes` (los tres CSV unidos), que no se modifica

### Fechas
Se convierten `fecha_creacion` y `fecha_cierre` de texto a formato de fecha, tomando en cuenta que vienen escritas de dos maneras (`dd/mm/aaaa` y `dd/mm/aa`).

In [71]:
# R1 (fechas): copia de trabajo y conversión de fecha_creacion y fecha_cierre de texto a fecha (dd/mm/aaaa y dd/mm/aa); se reportan los fallos.
limpio = incidentes.copy()

for col in ["fecha_creacion", "fecha_cierre"]:
    original = limpio[col].copy()
    f4 = pd.to_datetime(original, format="%d/%m/%Y", errors="coerce")
    f2 = pd.to_datetime(original, format="%d/%m/%y", errors="coerce")
    convertida = f4.fillna(f2)
    fallos = original.notna() & convertida.isna()
    print(col, "| fallos:", fallos.sum(), "| mín:", convertida.min(), "| máx:", convertida.max())
    limpio[col] = convertida

print("filas conservadas:", len(limpio) == len(incidentes))
print(limpio[["fecha_creacion", "fecha_cierre"]].dtypes)

fecha_creacion | fallos: 0 | mín: 2016-12-07 00:00:00 | máx: 2019-11-30 00:00:00
fecha_cierre | fallos: 0 | mín: 2017-01-01 00:00:00 | máx: 2019-11-30 00:00:00
filas conservadas: True
fecha_creacion    datetime64[ns]
fecha_cierre      datetime64[ns]
dtype: object


In [72]:
# R2 (horas): conversión de hora_creacion y hora_cierre (HH:MM:SS) a tipo hora; se reportan fallos.
for col in ["hora_creacion", "hora_cierre"]:
    original = limpio[col].copy()
    convertida = pd.to_datetime(original, format="%H:%M:%S", errors="coerce")
    fallos = original.notna() & convertida.isna()
    print(col, "| fallos de conversión:", fallos.sum())
    limpio[col] = convertida.dt.time

print("filas conservadas:", len(limpio) == len(incidentes))

hora_creacion | fallos de conversión: 0
hora_cierre | fallos de conversión: 0
filas conservadas: True


In [73]:
# R3 (texto): "Detención" pasa de mayúscula a minúscula en incidente_c4 (23 a 22 tipos); también se quitan espacios exteriores.
antes = limpio["incidente_c4"].nunique()
limpio["incidente_c4"] = limpio["incidente_c4"].astype("string").str.strip().str.lower()

print("tipos distintos | antes:", antes, "| después:", limpio["incidente_c4"].nunique())
print("detención ciudadana-accidente automovilístico:",
      (limpio["incidente_c4"] == "detención ciudadana-accidente automovilístico").sum())
print("filas conservadas:", len(limpio) == len(incidentes))

tipos distintos | antes: 23 | después: 22
detención ciudadana-accidente automovilístico: 80
filas conservadas: True


### Casos a revisar (sin corregir)
Se agregan tres columnas indicadoras (`True` o `False`) a la copia `limpio` para señalar coordenadas fuera del rango del diccionario, falsas alarmas (código F) y fechas de creación de otro año.

In [74]:
# Marcas sin modificar valores: coordenadas fuera del rango del diccionario, falsas alarmas (código F) y fecha de creación de otro año.
lat_ok = limpio["latitud"].between(19.09402002, 19.6322009)
lon_ok = limpio["longitud"].between(-99.37056146, -98.92945008)

limpio["coord_fuera_rango"] = (limpio["latitud"].notna() & ~lat_ok) | (limpio["longitud"].notna() & ~lon_ok)
limpio["es_falso_f"] = limpio["codigo_cierre"].str.startswith("(F)")
limpio["fecha_otro_anio"] = limpio["fecha_creacion"].dt.year != limpio["archivo_origen"]

print(limpio[["coord_fuera_rango", "es_falso_f", "fecha_otro_anio"]].sum())
print("archivo_origen distinto de año_cierre:", (limpio["año_cierre"] != limpio["archivo_origen"]).sum())
print("filas conservadas:", len(limpio) == len(incidentes))

coord_fuera_rango      13
es_falso_f           5244
fecha_otro_anio        49
dtype: int64
archivo_origen distinto de año_cierre: 0
filas conservadas: True


In [75]:
# Comprobación: los faltantes se conservan sin rellenar y no hay duplicados en la copia de trabajo.
columnas = ["delegacion_inicio", "delegacion_cierre", "latitud", "longitud", "geopoint"]
print(pd.DataFrame({"incidentes": incidentes[columnas].isna().sum(),
                    "limpio": limpio[columnas].isna().sum()}))

marcas = ["coord_fuera_rango", "es_falso_f", "fecha_otro_anio", "archivo_origen"]
print("duplicados exactos:", limpio.drop(columns=marcas).duplicated().sum())
print("folios repetidos:", limpio["folio"].duplicated().sum())

                   incidentes  limpio
delegacion_inicio          59      59
delegacion_cierre          41      41
latitud                   443     443
longitud                  435     435
geopoint                  446     446
duplicados exactos: 0
folios repetidos: 0


In [76]:
# R4 (alcaldía): se quitan de la copia las 59 filas sin alcaldía de inicio (56 vacías y 3 con "NA"); la pregunta necesita saber en qué alcaldía ocurrió cada incidente.
antes = len(limpio)
limpio = limpio.dropna(subset=["delegacion_inicio"]).copy()
print("filas antes:", antes, "| después:", len(limpio), "| excluidas:", antes - len(limpio))

filas antes: 680746 | después: 680687 | excluidas: 59


## 4. Validación
Objetivo: comprobar con condiciones concretas que los datos sirven para responder la pregunta de negocio. Se revisa que los campos estén completos, que el folio sea único y que los tipos, las categorías, las fechas y las coordenadas sean correctos.

In [77]:
# Validaciones: comprueba que los datos preparados cumplen las condiciones (campos completos, folio único, tipos, categorías, fechas, coordenadas y filas excluidas) e indica cuántos casos fallan en cada una.
import datetime as dt

# Catálogo: las 16 alcaldías de la CDMX. El diccionario documenta 17 valores; los "NA" (leídos como vacíos) se revisan como faltantes.
alcaldias = {"IZTAPALAPA", "GUSTAVO A. MADERO", "CUAUHTEMOC", "MIGUEL HIDALGO", "COYOACAN",
             "BENITO JUAREZ", "ALVARO OBREGON", "VENUSTIANO CARRANZA", "TLALPAN", "AZCAPOTZALCO",
             "IZTACALCO", "XOCHIMILCO", "TLAHUAC", "CUAJIMALPA", "MAGDALENA CONTRERAS", "MILPA ALTA"}
dias = {"Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"}
codigos = {"A", "D", "N", "F", "I"}
obligatorios = ["folio", "fecha_creacion", "hora_creacion", "dia_semana", "delegacion_inicio", "incidente_c4"]

def es_hora(columna):
    return (~limpio[columna].map(lambda x: isinstance(x, dt.time))).sum()

pruebas = {
    "Obligatorios sin vacíos": limpio[obligatorios].isna().any(axis=1).sum(),
    "Folio único": limpio["folio"].duplicated().sum(),
    "Fechas en tipo fecha": sum(not pd.api.types.is_datetime64_any_dtype(limpio[c]) for c in ["fecha_creacion", "fecha_cierre"]),
    "Horas en tipo hora": es_hora("hora_creacion") + es_hora("hora_cierre"),
    "Día de la semana en catálogo": (~limpio["dia_semana"].isin(dias)).sum(),
    "Alcaldía de inicio en catálogo": (limpio["delegacion_inicio"].notna() & ~limpio["delegacion_inicio"].isin(alcaldias)).sum(),
    "Alcaldía de cierre en catálogo": (limpio["delegacion_cierre"].notna() & ~limpio["delegacion_cierre"].isin(alcaldias)).sum(),
    "Código de cierre en catálogo": (~limpio["codigo_cierre"].str[1].isin(codigos)).sum(),
    "Fecha de creación en rango": (~limpio["fecha_creacion"].between("2013-12-31", "2022-07-31")).sum(),
    "Fecha de cierre en rango": (~limpio["fecha_cierre"].between("2014-01-01", "2022-07-31")).sum(),
    "Cierre no anterior a creación": (limpio["fecha_cierre"] < limpio["fecha_creacion"]).sum(),
    "Coordenadas en rango": limpio["coord_fuera_rango"].sum(),
        "Filas excluidas solo por R4 (59)": abs(len(incidentes) - len(limpio) - 59),
}

resultados = pd.DataFrame({"casos que no cumplen": pd.Series(pruebas)})
resultados["cumple"] = resultados["casos que no cumplen"] == 0
print(resultados)

                                  casos que no cumplen  cumple
Obligatorios sin vacíos                              0    True
Folio único                                          0    True
Fechas en tipo fecha                                 0    True
Horas en tipo hora                                   0    True
Día de la semana en catálogo                         0    True
Alcaldía de inicio en catálogo                       0    True
Alcaldía de cierre en catálogo                       0    True
Código de cierre en catálogo                         0    True
Fecha de creación en rango                           0    True
Fecha de cierre en rango                             0    True
Cierre no anterior a creación                        0    True
Coordenadas en rango                                13   False
Filas excluidas solo por R4 (59)                     0    True


### Comparación antes y después

Se compara el perfil inicial (`incidentes`) con la copia preparada (`limpio`) y se ve cuántas filas quedan para cada análisis.

In [83]:
# Compara los datos antes y después de limpiar, y cuenta cuántas filas sirven para cada tipo de análisis.
def resumen(t):
    return {
        "filas": len(t),
        "columnas": t.shape[1],
        "tipo de fecha_creacion": str(t["fecha_creacion"].dtype),
        "tipos distintos de incidente_c4": t["incidente_c4"].nunique(),
        "nulos en delegacion_inicio": t["delegacion_inicio"].isna().sum(),
        "folios repetidos": t["folio"].duplicated().sum(),
    }

print(pd.DataFrame({"antes (incidentes)": resumen(incidentes), "después (limpio)": resumen(limpio)}))

poblaciones = {
    "Todas las filas": len(limpio),
    "Con alcaldía de inicio": limpio["delegacion_inicio"].notna().sum(),
    "Sin falsas alarmas (código F)": (~limpio["es_falso_f"]).sum(),
    "Con alcaldía y sin falsas alarmas": (limpio["delegacion_inicio"].notna() & ~limpio["es_falso_f"]).sum(),
    "Con coordenadas completas y en rango": (limpio["latitud"].notna() & limpio["longitud"].notna() & ~limpio["coord_fuera_rango"]).sum(),
}
print(pd.Series(poblaciones))

                                antes (incidentes) después (limpio)
filas                                       680746           680687
columnas                                        19               22
tipo de fecha_creacion                      object   datetime64[ns]
tipos distintos de incidente_c4                 23               22
nulos en delegacion_inicio                      59                0
folios repetidos                                 0                0
Todas las filas                         680687
Con alcaldía de inicio                  680687
Sin falsas alarmas (código F)           675484
Con alcaldía y sin falsas alarmas       675484
Con coordenadas completas y en rango    680241
dtype: int64


### Comparación de las falsas alarmas
Dos columnas marcan falsas alarmas: `clas_con_f_alarma` (5,248) y `codigo_cierre` con código F (5,244). Aquí se compara si marcan las mismas filas y por qué no dan el mismo número.

In [79]:
# Cruce entre clas_con_f_alarma = "FALSA ALARMA" y codigo_cierre (F); se muestran las filas que no coinciden.
clas_falsa = limpio["clas_con_f_alarma"] == "FALSA ALARMA"
codigo_f = limpio["es_falso_f"]

print(pd.crosstab(clas_falsa, codigo_f, rownames=["clas = FALSA ALARMA"], colnames=["codigo = F"]))

no_coinciden = limpio.loc[clas_falsa != codigo_f, ["folio", "clas_con_f_alarma", "codigo_cierre"]].copy()
no_coinciden["codigo_cierre"] = no_coinciden["codigo_cierre"].str[:3]
print(no_coinciden.head(20))

codigo = F            False  True 
clas = FALSA ALARMA               
False                675479      1
True                      5   5202
                  folio clas_con_f_alarma codigo_cierre
37547   IZ/170101/05724      FALSA ALARMA           (A)
38032   CH/170105/03406      FALSA ALARMA           (A)
147974  CH/170107/03746      FALSA ALARMA           (N)
148763  C5/170112/03100      FALSA ALARMA           (A)
205680  CH/170103/00573      FALSA ALARMA           (A)
369470  C5/180131/03392        EMERGENCIA           (F)


In [80]:
# Cuenta los vacíos por columna en limpio; tras la R4 solo quedan coordenadas (latitud, longitud, geopoint).
faltantes = limpio.isna().sum()
print(faltantes[faltantes > 0])

latitud     443
longitud    435
geopoint    446
dtype: int64
